# SLAVA v0: collection and review of 112 simulation scenes

Этот notebook собирает environment-first inventory:

- 30 LIBERO tasks × init states `[0, 17, 34]` = 90 сцен;
- 4 SimplerEnv bridge tasks: по 3 базовых episode IDs `[0, 8, 16]` и по 5 дополнительных `[1, 4, 12, 20, 23]` для `widowx_carrot_on_plate` и `widowx_stack_cube` = 22 сцены;
- всего 112 строк `task × init_state`.

Симуляторы запускаются в отдельных virtual environments. Notebook хранит объединённый DataFrame, интерактивную разметку и экспорт JSONL/CSV. Сбор работает в resume-режиме.

In [7]:
from pathlib import Path
import json
import importlib
import os
import shutil
import subprocess
import sys
import pandas as pd
from IPython.display import display

In [8]:


def find_project_root():
    candidates = [os.environ.get('SLAVA_ROOT'), Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if not candidate:
            continue
        candidate = Path(candidate).expanduser().resolve()
        if (candidate / 'src' / 'slava_inventory').is_dir():
            return candidate
    raise RuntimeError('Не найден корень SLAVA_dev. Откройте папку проекта в VS Code перед запуском ноутбука.')

PROJECT_ROOT = find_project_root()
DEPS_DIR = Path(os.environ.get('SLAVA_DEPS_DIR', PROJECT_ROOT.parent)).expanduser().resolve()
LIBERO_REPO = Path(os.environ.get('LIBERO_ROOT', DEPS_DIR / 'LIBERO')).expanduser().resolve()
SIMPLER_REPO = Path(os.environ.get('SIMPLERENV_ROOT', DEPS_DIR / 'SimplerEnv')).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import slava_inventory.notebook_ui as notebook_ui
notebook_ui = importlib.reload(notebook_ui)
InventoryReviewer = notebook_ui.InventoryReviewer
LexiconReviewer = notebook_ui.LexiconReviewer
VisibilityReviewer = notebook_ui.VisibilityReviewer
create_or_update_lexicon = notebook_ui.create_or_update_lexicon
load_lexicon_dataframe = notebook_ui.load_lexicon_dataframe
export_inventory_dataframe = notebook_ui.export_inventory_dataframe
merge_inventories = notebook_ui.merge_inventories

conda_candidates = [
    os.environ.get('CONDA_EXE'),
    shutil.which('conda'),
    '/opt/miniforge3/bin/conda',
    '/opt/conda/bin/conda',
]
CONDA_EXE = next((str(Path(p)) for p in conda_candidates if p and Path(p).is_file()), None)
LIBERO_CONDA_ENV = 'slava-libero'
SIMPLER_CONDA_ENV = 'slava-simpler'
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Conda:', CONDA_EXE or 'NOT FOUND')

Project: /Users/alexkarachun/Documents/DEV/SLAVA_dev
Data: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data
Conda: /opt/miniconda3/bin/conda


## 1. Проверка путей и окружений

Если Conda environment отсутствует, сначала выполните инструкции из `README.md`.

In [9]:
# assert PROJECT_ROOT.exists(), PROJECT_ROOT
# assert LIBERO_REPO.exists(), LIBERO_REPO
# assert SIMPLER_REPO.exists(), SIMPLER_REPO

if CONDA_EXE is None:
    print('WARNING: Conda executable was not found. See README.md.')
    conda_envs = {}
else:
    result = subprocess.run(
        [CONDA_EXE, 'env', 'list', '--json'], capture_output=True, text=True, check=True
    )
    env_paths = json.loads(result.stdout)['envs']
    conda_envs = {Path(path).name: path for path in env_paths}
    display(pd.DataFrame(sorted(conda_envs.items()), columns=['environment', 'path']))

for env_name in [LIBERO_CONDA_ENV, SIMPLER_CONDA_ENV]:
    if env_name not in conda_envs:
        print(f'WARNING: Conda environment {env_name!r} was not found')

,environment,path
0,base,/opt/homebrew/Caskroom/miniconda/base
1,miniconda3,/opt/miniconda3
2,selfmade-diffusion,/opt/miniconda3/envs/selfmade-diffusion
3,slava-simpler,/opt/homebrew/Caskroom/miniconda/base/envs/sla...
4,through_guidance,/opt/miniconda3/envs/through_guidance


## 2. Сбор сцен

По умолчанию collectors не запускаются. Поменяйте нужный флаг на `True`.

- Повторный запуск безопасен: существующие `task_uid` пропускаются.
- `OVERWRITE_EXISTING=True` полностью пересоздаст соответствующий partial inventory.
- Ошибки отдельных сцен сохраняются в `data/collection_errors.jsonl`.
- Для LIBERO headless renderer запускается с `MUJOCO_GL=egl`.

In [10]:
RUN_LIBERO = False
RUN_SIMPLER = False
OVERWRITE_EXISTING = False
FAIL_FAST = True

def run_streaming(command, extra_env=None):
    env = None
    if extra_env:
        import os
        env = os.environ.copy()
        env.update(extra_env)
    print(' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, env=env, check=True)

if (RUN_LIBERO or RUN_SIMPLER) and CONDA_EXE is None:
    raise RuntimeError('Conda executable was not found. See README.md.')

if RUN_LIBERO:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', LIBERO_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_libero.py',
        '--libero-repo', LIBERO_REPO,
        '--output-root', DATA_DIR,
        '--init-state-ids', 0, 17, 34,
        '--image-size', 256,
        '--settle-steps', 0,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd, {'MUJOCO_GL': 'egl', 'MUJOCO_EGL_DEVICE_ID': '0'})

if RUN_SIMPLER:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', SIMPLER_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_simpler.py',
        '--simpler-repo', SIMPLER_REPO,
        '--output-root', DATA_DIR,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd)

## 3. Объединение partial inventories

При повторном объединении ручные annotations из существующего `task_inventory.jsonl` сохраняются.

In [11]:
inventory_df = merge_inventories(DATA_DIR)
print('Total scenes:', len(inventory_df))
# Reload the module so rerunning this section cannot write an obsolete CSV schema.
notebook_ui = importlib.reload(notebook_ui)
if inventory_df.empty:
    print('Inventory is empty. Set RUN_LIBERO/RUN_SIMPLER=True and run the collection cell.')
else:
    display(inventory_df.groupby('suite').size().rename('scenes').to_frame())
    display(inventory_df[['task_uid', 'suite', 'canonical_en', 'usable_for_slava']].head())
    if len(inventory_df) != 112:
        print('WARNING: expected 112 rows. Inspect data/collection_errors.jsonl and rerun collectors.')

Total scenes: 112


,scenes
suite,
libero_goal,30
libero_object,30
libero_spatial,30
simpler_bridge,22


,task_uid,suite,canonical_en,usable_for_slava
0,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,True
1,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,False
2,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,True
3,libero_goal__open_the_top_drawer_and_put_the_b...,libero_goal,open the top drawer and put the bowl inside,False
4,libero_goal__open_the_top_drawer_and_put_the_b...,libero_goal,open the top drawer and put the bowl inside,False


## 4. Быстрая проверка видимости benchmark-объектов

Форма показывает одну сцену, обе доступные камеры и все объекты из канонического `objects_raw`.

Для каждого объекта выберите `visible`, `visible_partial` или `not visible`. `visible_partial` означает, что объект виден лишь частично, но его всё ещё можно распознать. У SimplerEnv нет wrist camera, поэтому `visible_wrist` остаётся `null`. Кнопка **Save + next** сразу атомарно сохраняет весь DataFrame в `data/task_inventory.jsonl` и открывает следующую незаполненную сцену. `?` означает, что поле ещё не проверено; такая сцена остаётся в фильтре **Only unfinished**. Кнопки **All visible** ускоряют разметку сцен, где все объекты хорошо видны.

In [12]:
visibility_reviewer = None
if inventory_df.empty:
    print('Visibility review is unavailable until the inventory has been collected.')
else:
    visibility_reviewer = VisibilityReviewer(inventory_df, DATA_DIR)
    visibility_reviewer.show()

visible - целевую зону объекта видно целиком

partial - видно только часть объекта (возможно целевую зону объекта не видно)

not visible - объекта нет на изображении


Сводка прогресса (можно выполнять повторно в любой момент):

In [13]:
if visibility_reviewer is None:
    print('No visibility-review summary yet.')
else:
    inventory_df = visibility_reviewer.df
    finished = sum(visibility_reviewer._scene_complete(i) for i in range(len(inventory_df)))
    print(f'Visibility finished: {finished}/{len(inventory_df)}')
    rows = []
    for suite, indices in inventory_df.groupby('suite').groups.items():
        rows.append({
            'suite': suite,
            'finished': sum(visibility_reviewer._scene_complete(i) for i in indices),
            'total': len(indices),
        })
    display(pd.DataFrame(rows).set_index('suite'))

Visibility finished: 19/112


,finished,total
suite,,
libero_goal,0,30
libero_object,0,30
libero_spatial,0,30
simpler_bridge,19,22


## 5. Полный интерактивный review сцен

Для каждой сцены можно:

- отметить `usable_for_slava`;
- записать заметки;
- заполнить candidate semantic slots;
- вручную отметить видимость каждого sim object в agent/wrist view.

Кнопки Previous/Next сначала сохраняют текущую форму в DataFrame. Кнопка Export JSONL атомарно записывает DataFrame в `data/task_inventory.jsonl`.

In [14]:
scene_reviewer = None
if inventory_df.empty:
    print('Scene review is unavailable until the inventory has been collected.')
else:
    scene_reviewer = InventoryReviewer(inventory_df, DATA_DIR)
    scene_reviewer.show()

После работы с формой используйте DataFrame из reviewer:

In [15]:
if scene_reviewer is None:
    print('No scene-review summary yet.')
else:
    inventory_df = scene_reviewer.df
    summary = pd.DataFrame({
        'value': [
            len(inventory_df),
            int(inventory_df.usable_for_slava.notna().sum()),
            int((inventory_df.usable_for_slava == True).sum()),
        ]
    }, index=['total', 'reviewed', 'usable'])
    display(summary)

,value
total,112
reviewed,61
usable,37


## 6. Создание object_lexicon.csv

Список строится из уникальных `raw_name` всех собранных sim objects. Существующие русские annotations не перезаписываются при повторном запуске. Технические невидимые объекты вроде `dummy_sink_target_plane` исключаются.

In [16]:
if inventory_df.empty:
    lexicon_df = pd.DataFrame()
    print('Object lexicon is unavailable until the inventory has been collected.')
else:
    lexicon_df = notebook_ui.create_or_update_lexicon(DATA_DIR, inventory_df)
    print('Unique lexicon objects:', len(lexicon_df))
    display(lexicon_df.head(20))

Unique lexicon objects: 27


,raw_name,category_en,category_ru,semantic_subtype_en,semantic_subtype_ru,canonical_name_en,canonical_name_ru,visual_attributes_en,visual_attributes_ru,semantic_identity_visually_recoverable,color_en,color_ru,allowed_synonyms_ru,usable_v0,notes
0,akita_black_bowl,bowl,миска,ceramic bowl,керамическая миска,black bowl,черная миска,black round bowl,черная круглая миска,yes,black,черная,пиала,yes,
1,alphabet_soup,can,банка,soup,суп,soup can,банка супа,blue-orange can,сине-оранжевая банка,no,blue,синяя,консервная банка супа,yes,
2,baked_green_cube_3cm,cube,кубик,cube,кубик,green cube,зеленый кубик,small green cube,маленький зеленый кубик,yes,green,зеленый,куб,yes,
3,baked_yellow_cube_3cm,cube,кубик,cube,кубик,yellow cube,желтый кубик,small yellow cube,маленький желтый кубик,yes,yellow,желтый,куб,yes,
4,basket,basket,корзина,woven basket,плетеная корзина,basket,корзина,beige woven basket,бежевая плетеная корзина,yes,beige,бежевая,корзинка,yes,
5,bbq_sauce,bottle,бутылка,barbecue sauce,соус барбекю,barbecue sauce bottle,бутылка соуса барбекю,brown bottle with an orange cap,коричневая бутылка с оранжевой крышкой,no,brown,коричневая,бутылка с соусом барбекю,yes,
6,bridge_carrot_generated_modified,vegetable,овощ,carrot,морковь,carrot,морковь,orange carrot,оранжевая морковь,yes,orange,оранжевая,морковка,yes,
7,bridge_plate_objaverse_larger,plate,тарелка,dinner plate,столовая тарелка,yellow plate,желтая тарелка,large pale yellow plate,большая светло-желтая тарелка,yes,yellow,желтая,столовая тарелка,yes,
8,bridge_spoon_generated_modified,cutlery,столовый прибор,spoon,ложка,green-handled spoon,ложка с зеленой ручкой,white spoon with a green handle,белая ложка с зеленой ручкой,yes,green,зеленая,ложечка,yes,
9,butter,brick pack,брикет,butter,сливочное масло,butter package,брикет масла,red brick pack with a blue cow,красный брикет с синей коровой,no,red,красный,упаковка масла,yes,


## 7. Интерактивный review лексикона

`usable_v0 = no` ставьте, если объект трудно опознать на рендере или нельзя естественно и однозначно назвать по-русски.

In [17]:
# Always reload code and data: the kernel may still hold the legacy 8-column schema.
notebook_ui = importlib.reload(notebook_ui)
lexicon_reviewer = None
lexicon_path = DATA_DIR / 'object_lexicon.csv'
if not lexicon_path.is_file():
    print('Lexicon review is unavailable until object_lexicon.csv has been created.')
else:
    lexicon_df = notebook_ui.load_lexicon_dataframe(lexicon_path)
    lexicon_reviewer = notebook_ui.LexiconReviewer(lexicon_df, lexicon_path)
    lexicon_reviewer.show()

## 8. Финальный экспорт и sanity checks

In [18]:
if scene_reviewer is None:
    print('Nothing to export yet: inventory is empty.')
else:
    inventory_df = scene_reviewer.df
    export_inventory_dataframe(inventory_df, DATA_DIR / 'task_inventory.jsonl')
    duplicate_uids = inventory_df.task_uid[inventory_df.task_uid.duplicated()].tolist()
    missing_agent_images = [
        row.task_uid for row in inventory_df.itertuples()
        if not (DATA_DIR / row.images['agentview_rgb']).exists()
    ]
    missing_wrist_images = [
        row.task_uid for row in inventory_df.itertuples()
        if row.source['environment'] == 'LIBERO'
        and not (DATA_DIR / row.images['wrist_rgb']).exists()
    ]
    print('Rows:', len(inventory_df))
    print('Duplicate task_uid:', duplicate_uids)
    print('Missing agent images:', missing_agent_images)
    print('Missing required LIBERO wrist images:', missing_wrist_images)
    print('Saved:', DATA_DIR / 'task_inventory.jsonl')
    print('Saved:', DATA_DIR / 'object_lexicon.csv')

Rows: 112
Duplicate task_uid: []
Missing agent images: []
Missing required LIBERO wrist images: []
Saved: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data/task_inventory.jsonl
Saved: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data/object_lexicon.csv


## 9. Dashboard исключения неудачных сцен

Dashboard автоматически оставляет только сцены, где:

- минимальная уже размеченная `objects_raw.visible_agentview` равна `true`;
- минимальная уже размеченная `objects_raw.visible_wrist` не хуже `visible_partial`;
- каждый объект имеет `object_lexicon.usable_v0 = yes`.

Значения visibility `null` не участвуют в вычислении минимума — это соответствует фильтрам screenshot sheet. Для SimplerEnv отсутствие wrist-камеры считается `N/A` и не исключает сцену.

Все показанные сцены по умолчанию считаются удачными. Ставьте галочку **«Исключить сцену»** только на неудачных карточках — отмеченная карточка сразу исчезнет. Кнопка **«← Вернуть последнюю»** отменяет последнее исключение и возвращает карточку на исходное место. При сохранении все оставшиеся сцены получают `usable_for_slava = true`, исключенные — `false`, а не прошедшие текущие фильтры — `null`. Благодаря этому раздел квот видит ровно сохраненный здесь набор удачных сцен. Dashboard показывает, сколько сцен останется и сколько еще нужно исключить до ориентира 20. Клик в любом месте карточки переключает галочку.

In [19]:
# Reload from disk so the dashboard always uses the latest saved review state.
notebook_ui = importlib.reload(notebook_ui)
selection_df = notebook_ui.load_inventory_dataframe(DATA_DIR)
selection_lexicon_df = notebook_ui.load_lexicon_dataframe(
    DATA_DIR / 'object_lexicon.csv'
)

top20_selector = notebook_ui.TopSceneSelector(
    selection_df,
    selection_lexicon_df,
    DATA_DIR,
)
top20_selector.show()

## 10. Разметка применимости квот

Dashboard показывает только сцены с `usable_for_slava = true`. Для каждой сцены заполните девять фиксированных полей `quota_eligibility`: **Подходит**, **Не подходит** или **?**. Все `null` по умолчанию отображаются как **Не подходит**. Если следующая сцена относится к той же задаче, но имеет другой init/episode, её незаполненные поля наследуют ответы предыдущей сцены. Уже сохранённые значения не перезаписываются. Кнопка `Save + next` сразу сохраняет текущую сцену в `task_inventory.jsonl`.

In [22]:
notebook_ui = importlib.reload(notebook_ui)
quota_df = notebook_ui.load_inventory_dataframe(DATA_DIR)
quota_reviewer = notebook_ui.QuotaEligibilityReviewer(quota_df, DATA_DIR)
quota_reviewer.show()

## 11. Проверка выполнимости всех квот

Ячейка ищет точное подмножество из 20 допущенных сцен, которое одновременно выполняет все девять минимумов. Если существует несколько комбинаций, выбирается комбинация с минимальным числом SimplerEnv-сцен. Если хотя бы одна допущенная сцена размечена не полностью, результат остается `INCOMPLETE`.

In [23]:
quota_df = notebook_ui.load_inventory_dataframe(DATA_DIR)
quota_result = notebook_ui.evaluate_quota_feasibility(quota_df, target_size=20)

quota_summary = pd.DataFrame([
    {
        'quota': field,
        'description': notebook_ui.QUOTA_LABELS[field],
        'minimum': minimum,
        'eligible': quota_result['counts'][field],
        'pending': quota_result['pending'][field],
        'enough_in_full_pool': quota_result['counts'][field] >= minimum,
    }
    for field, minimum in notebook_ui.QUOTA_REQUIREMENTS.items()
])
display(quota_summary)

if quota_result['incomplete_task_uids']:
    print('INCOMPLETE: полностью разметьте quota_eligibility для всех допущенных сцен.')
    print('Incomplete scenes:', len(quota_result['incomplete_task_uids']))
elif quota_result['feasible']:
    print('FEASIBLE: существует набор из 20 сцен, одновременно выполняющий все квоты.')
    print('SimplerEnv scenes in preferred set:', quota_result['witness_simpler_count'])
    display(pd.DataFrame({
        'scene_number': range(1, 21),
        'task_uid': quota_result['witness_task_uids'],
    }))
else:
    print('NOT FEASIBLE: среди допущенных сцен нет набора из 20, выполняющего все квоты.')

,quota,description,minimum,eligible,pending,enough_in_full_pool
0,spatial_relation,Spatial relation: left/right/on/next_to,8,19,0,True
1,pick_with_distractors,Pick / object selection среди distractors,5,21,0,True
2,container,Container: put X in drawer/bowl/basket/sink,4,10,0,True
3,surface,Surface: put X on plate/tray/table,3,6,0,True
4,has_distractor,Есть distractor,10,25,0,True
5,same_category_distractor,Same-category distractor,5,17,0,True
6,same_color_distractor,Same-color distractor,5,12,0,True
7,ru_case_swap,ru_case_swap / role-stress,6,8,0,True
8,ru_negation,ru_negation,12,25,0,True


FEASIBLE: существует набор из 20 сцен, одновременно выполняющий все квоты.
SimplerEnv scenes in preferred set: 9


,scene_number,task_uid
0,1,libero_goal__put_the_wine_bottle_on_the_rack__...
1,2,libero_object__pick_up_the_butter_and_place_it...
2,3,libero_object__pick_up_the_butter_and_place_it...
3,4,libero_object__pick_up_the_butter_and_place_it...
4,5,libero_object__pick_up_the_cream_cheese_and_pl...
5,6,libero_object__pick_up_the_cream_cheese_and_pl...
6,7,libero_object__pick_up_the_cream_cheese_and_pl...
7,8,libero_object__pick_up_the_milk_and_place_it_i...
8,9,libero_object__pick_up_the_tomato_sauce_and_pl...
9,10,libero_object__pick_up_the_tomato_sauce_and_pl...
